# MP08 — Frequency-Domain Image Explorer

Expanded engineering lesson; learner mastery unassessed.

## Intuition, definitions and worked mathematics
A discrete Fourier transform decomposes a 2D image into periodic spatial-frequency components. For H rows and W columns, F[u,v]=sum over h,w of x[h,w] exp(−2 pi i(uh/H+vw/W)). Here i²=−1, u and v are integer frequency-bin indices, and x is a single band. The forward NumPy FFT has no normalization; the inverse divides by HW.

Magnitude |F| describes frequency strength. Phase describes placement; magnitude alone cannot reconstruct the original image. FFT assumes periodic boundaries, so a mismatch between opposite edges can add spectral energy. fftshift moves the zero-frequency component to the display center. We display log(1+|F|) because the DC component may dominate; this visualization is not a physical logarithmic radiance measurement.

For a constant 4×4 image of value 2, F[0,0]=32 and all other bins are zero up to floating-point error. Parseval's relation is sum(x²)=sum(|F|²)/(HW) under this normalization. The inverse must reproduce x within numerical precision.

Gaussian smoothing suppresses high spatial frequencies. White noise adds broad frequency content. A spectrum alone does not identify which structures are physically real; textures, noise and boundary discontinuities can overlap in frequency.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp08_tools import gaussian,filter_hwc,degrade_hwc,spectrum
x=np.full((4,4),2.); f,log=spectrum(x)
assert f[0,0]==32
np.testing.assert_allclose(np.fft.ifft2(f).real,x)
assert np.isclose((x*x).sum(),(np.abs(f)**2).sum()/x.size)
wave=np.cos(2*np.pi*np.arange(16)/4)[None,:].repeat(16,0)
f,_=spectrum(wave); print('dominant column frequency bin:',np.argmax(np.abs(f[0,1:]))+1)

## Real-data experiment and interpretation
The experiment compares the same real training red band, its Gaussian-blurred version, and a seeded noisy version. All frequency displays share fixed limits. It records inverse reconstruction error and both sides of Parseval's identity. Changes in high-frequency magnitude are visible, but this does not validate an SR model or distinguish useful detail from invented detail.

Run `python scripts/degradation_experiments.py` explicitly from the repository root. The notebook below reads recorded measurements; it does not download data or execute a hidden research run. The source is a checksummed 64×64 NAIP training crop with fixed stored-value/10000 scaling, not verified reflectance.

In [ ]:
p=root/'reports/mp05-mp08-experiments.json'
if p.exists():
    record=json.loads(p.read_text())
    print(record['mp08'])
else: print('Real experiment has not run in this checkout.')

In [ ]:
from IPython.display import Image,display
p=root/'reports'/'mp08-frequency.png'
if p.exists(): display(Image(filename=str(p)))

## Practice — attempt before reading solutions
1. What is the DC value of an 8×8 constant image valued 0.5?
2. Why does shifting an image affect phase?
3. Why not compare independently auto-scaled spectral plots?

## Separate solution guide
1. 64×0.5=32 under the unnormalized forward transform.
2. A translation changes the alignment of each periodic component while preserving magnitude for a circular shift.
3. Different color scales can make weak and strong components look equally intense.

These are generated examples, not evidence that a learner solved the exercises.

In [ ]:
x=np.random.default_rng(17).random((8,8)); f,_=spectrum(x); shifted,_=spectrum(np.roll(x,1,axis=1))
np.testing.assert_allclose(np.abs(f),np.abs(shifted),atol=1e-12)
assert spectrum(np.full((8,8),.5))[0][0,0]==32

## Reuse, execution and limitations
Use `src/rssr/degradation_lab.py` and `python -m pytest tests/test_degradation_lab.py`. Follow [local and Colab setup](../../docs/setup.md); run from notebooks/ or the repository root. The shared NumPy code preserves HWC channels and uses explicit boundaries. No physical sensor calibration or geographic generalization is established.

References: the authoritative stage brief, [dataset card](../../docs/dataset-cards.md), [protocol](../../docs/protocol.md), and [NumPy FFT documentation](https://numpy.org/doc/stable/reference/routines.fft.html). See [execution evidence](../../reports/notebook-execution.json) and [experiment record](../../reports/mp05-mp08-experiments.json).